In [1]:
# -----------------------------
# Imports & Environment Setup
# -----------------------------
import os                          # Access environment variables (like API keys)
from dotenv import load_dotenv     # Load variables from a .env file into environment

# Typing utilities for structured state definitions
from typing import Annotated, TypedDict, Sequence

# -----------------------------
# LangGraph Core (workflow + agent)
# -----------------------------
from langgraph.prebuilt import create_react_agent   # Prebuilt ReAct agent (Reason + Act loop)
from langgraph.graph import END, StateGraph         # Graph builder + END marker
from langgraph.graph.message import add_messages    # Utility to manage conversation messages

# -----------------------------
# LangChain Core (messages + tools)
# -----------------------------
from langchain_core.messages import BaseMessage, HumanMessage   # Chat message types
from langchain_core.tools import Tool                          # Define external tools agent can use

# -----------------------------
# Document Handling (load + split)
# -----------------------------
from langchain_community.document_loaders import WebBaseLoader  # Load webpage content as documents
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Split text into chunks

# -----------------------------
# Vector Store & Embeddings
# -----------------------------
from langchain_community.vectorstores import FAISS              # FAISS vector DB for similarity search
from langchain_community.embeddings import HuggingFaceEmbeddings # Convert text chunks into embeddings

# -----------------------------
# LLM (Groq backend)
# -----------------------------
from langchain_groq import ChatGroq   # Connect to Groq-hosted LLMs (e.g., Llama models)

# -----------------------------
# Edge Browser Tabs Metadata
# -----------------------------
# User's Edge browser tabs metadata. The tab with `isCurrent=true` is the active tab,
# while `isCurrent=false` are background tabs

c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# Load IPL points table page
docs = WebBaseLoader("https://www.iplt20.com/matches/points-table").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [3]:
# -----------------------------
# Embedding + Vector Store Setup
# -----------------------------

# Initialize HuggingFace embeddings model (pretrained sentence-transformers)
# This converts text chunks into numerical vectors for semantic search
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Build FAISS vector store from the split documents
# FAISS stores (text, vector) pairs and allows fast similarity search
vectorstore = FAISS.from_documents(chunks, embedding)

# Create a retriever interface from the vector store
# Retriever is used later to fetch the most relevant chunks for a given query
retriever = vectorstore.as_retriever()


C:\Users\admin\AppData\Local\Temp\ipykernel_12204\2876648972.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [4]:
# Define a function that acts as a retriever tool for cricket data
def cricket_retriever_tool(query: str) -> str:
    # Print a log message so we know when this tool is being used
    print("🏏 Using CricketRetriever tool")
    
    # Use the retriever to fetch relevant documents based on the query
    docs = retriever.invoke(query)
    
    # If no documents are found, return a fallback message
    if not docs:
        return f"No cricket info found for: {query}"
    
    # Otherwise, join the page content from all retrieved docs into one string
    return "\n".join([doc.page_content for doc in docs])

# Wrap the retriever function into a LangChain Tool object
cricket_tool = Tool(
    name="CricketRetriever",   # Name of the tool (used by the agent)
    description="Use this tool to fetch IPL points table info",  # Short description for the agent
    func=cricket_retriever_tool   # The function that will be executed when the tool is called
)

# Print the tool's name to confirm it was created successfully
print(cricket_tool.name)

CricketRetriever


In [5]:
# --------------------------
# 2. Initialize Groq LLM
# --------------------------
load_dotenv()
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [6]:
# --------------------------
# 3. Define the Agent Node
# --------------------------

# Create a list of tools that the agent can use.
# Here we only have one tool: the CricketRetriever.
tools = [cricket_tool]

# Initialize a ReAct-style agent node using LangGraph's helper.
# This wraps the LLM (Groq in our case) together with the tools,
# so the agent can decide when to call the tool and when to answer directly.
react_node = create_react_agent(llm, tools)


C:\Users\admin\AppData\Local\Temp\ipykernel_12204\1006206726.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_node = create_react_agent(llm, tools)


In [7]:
# --------------------------
# 4. LangGraph Agent State
# --------------------------

# Define a TypedDict class to represent the agent's state.
# This state will hold the conversation messages as the agent runs.
class AgentState(TypedDict):
    # 'messages' is a sequence (list) of BaseMessage objects (HumanMessage, AIMessage, etc.)
    # Annotated with 'add_messages' reducer so that new messages are appended
    # instead of overwriting the old ones. This keeps the full conversation history.
    messages: Annotated[Sequence[BaseMessage], add_messages]


In [8]:
# --------------------------
# 5. Build LangGraph Graph
# --------------------------

# Initialize a new StateGraph object using the AgentState definition.
# This graph will control the flow of nodes (steps) in our agent pipeline.
builder = StateGraph(AgentState)

# Add a node to the graph called "react_agent".
# This node represents the ReAct agent we created earlier (LLM + tools).
builder.add_node("react_agent", react_node)

# Set the entry point of the graph to "react_agent".
# This means execution will start from this node when the graph runs.
builder.set_entry_point("react_agent")

# Add an edge from "react_agent" to END.
# END is a special marker that tells LangGraph where the workflow finishes.
builder.add_edge("react_agent", END)

# Compile the graph into an executable object.
# After compilation, we can invoke the graph with input state and get results.
graph = builder.compile()


In [9]:
# --------------------------
# 6. Run the ReAct Agent
# --------------------------

# Standard Python entry point check.
# This ensures the code inside runs only when the script is executed directly,
# not when it is imported as a module.
if __name__ == "__main__":
    
    # Define the user's query (the question we want the agent to answer).
    user_query = "Show me the IPL points table and explain the top teams."
    
    # Create the initial state for the graph.
    # The state contains a list of messages, starting with a HumanMessage (the user input).
    state = {"messages": [HumanMessage(content=user_query)]}
    
    # Invoke the graph with the initial state.
    # This runs the agent node, which uses the LLM and tools to generate a response.
    result = graph.invoke(state)

    # Print the final answer from the agent.
    # The agent's response is stored in the last message of the state.
    print("\n Final Answer:\n", result["messages"][-1].content)


🏏 Using CricketRetriever tool

 Final Answer:
 The IPL points table is a ranking system that shows the performance of each team in the Indian Premier League. The top teams are determined by the number of points they have earned, which is calculated based on their wins, losses, and ties.

Here's a general explanation of the top teams in the IPL points table:

1. **Chennai Super Kings**: They are one of the most successful teams in the IPL, with three titles to their name. They have a strong squad and a good balance of experienced players and young talent.
2. **Mumbai Indians**: They are the most successful team in the IPL, with five titles. They have a strong squad and a good balance of experienced players and young talent.
3. **Gujarat Titans**: They are a relatively new team, but they have made a strong impact in their first few seasons. They have a good balance of experienced players and young talent.
4. **Lucknow Super Giants**: They are another relatively new team, but they have ma

In [10]:
# --------------------------
# 6. Run the ReAct Agent
# --------------------------

# Standard Python entry point check.
# This ensures the code inside runs only when the script is executed directly,
# not when it is imported as a module.
if __name__ == "__main__":
    
    # Define the user's query (the question we want the agent to answer).
    user_query = "what is temp today?."
    
    # Create the initial state for the graph.
    # The state contains a list of messages, starting with a HumanMessage (the user input).
    state = {"messages": [HumanMessage(content=user_query)]}
    
    # Invoke the graph with the initial state.
    # This runs the agent node, which uses the LLM and tools to generate a response.
    result = graph.invoke(state)

    # Print the final answer from the agent.
    # The agent's response is stored in the last message of the state.
    print("\n Final Answer:\n", result["messages"][-1].content)



 Final Answer:
 I'm not aware of any information about the current temperature.
